# 01. Design Problems: Toolbox and Basic Designs

"Design a data structure with operations X, Y, Z, each in O(1) / O(log n)." These test whether you can **combine** basic structures.

**Learning order:** APPROACH → TOOLBOX → RANDOMIZED SET → TIME-BASED KEY-VALUE STORE → LOGGER RATE LIMITER → HASHMAP FROM SCRATCH → INTERVIEW THINKING

## 1. Approach (5 steps)

1. **Clarify the API:** list every operation, its input/output, and edge cases (duplicates, missing keys).
2. **State the required complexity** of each operation.
3. **Pick structures** so that each operation is satisfied (a table below).
4. **Define the invariants** (what is always true: "the list and the dict always agree").
5. Code, then test the operations in a tricky order.

## 2. Toolbox

| Need | Structure |
|---|---|
| O(1) lookup by key | hash map |
| O(1) add/remove at both ends | deque |
| O(1) remove a known node, ordering by recency | hash map + doubly linked list (LRU) |
| min/max quickly | heap |
| sorted order, floor/ceiling, range | sorted list + `bisect` / balanced BST |
| prefix queries | trie |
| O(1) random element | **array** (+ hash map for index) |
| counts in a window of time | deque of timestamps |

## 3. Insert, Delete, GetRandom in O(1)

- `getRandom` needs a **list** (random index).
- `insert`/`remove` by value in O(1) needs a **dict** (value → index in the list).
- **Trick for O(1) delete:** swap the element with the **last** element, update that element's index in the dict, `pop()` the last. Never delete from the middle of a list.

In [1]:
import random

class RandomizedSet:
    def __init__(self):
        self.items = []
        self.pos = {}                                # value -> index in items

    def insert(self, val):
        if val in self.pos:
            return False
        self.pos[val] = len(self.items)
        self.items.append(val)
        return True

    def remove(self, val):
        if val not in self.pos:
            return False
        idx, last = self.pos[val], self.items[-1]
        self.items[idx] = last                       # move the last element into the hole
        self.pos[last] = idx
        self.items.pop()
        del self.pos[val]                            # delete AFTER updating (val may equal last)
        return True

    def get_random(self):
        return random.choice(self.items)

rs = RandomizedSet()
print(rs.insert(1), rs.remove(2), rs.insert(2))   # True False True
print(rs.get_random() in (1, 2), rs.remove(1), rs.insert(2), rs.get_random())   # True True False 2

True False True
True True False 2


## 4. Time Based Key-Value Store

`set(key, value, timestamp)` and `get(key, timestamp)` = the value with the **largest timestamp ≤ the query**. Timestamps for a key arrive in increasing order, so each key keeps a **sorted list**, and `get` uses **binary search** (`bisect_right`). Real life: version history, feature flags over time.

In [2]:
from bisect import bisect_right
from collections import defaultdict

class TimeMap:
    def __init__(self):
        self.times = defaultdict(list)               # key -> [timestamps]
        self.values = defaultdict(list)              # key -> [values]

    def set(self, key, value, timestamp):
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key, timestamp):
        i = bisect_right(self.times[key], timestamp)
        return self.values[key][i - 1] if i else ""

tm = TimeMap()
tm.set("foo", "bar", 1)
print(tm.get("foo", 1), tm.get("foo", 3), repr(tm.get("foo", 0)))   # bar bar ''
tm.set("foo", "bar2", 4)
print(tm.get("foo", 4), tm.get("foo", 5))   # bar2 bar2

bar bar ''
bar2 bar2


## 5. Logger Rate Limiter

Print a message only if it was **not printed in the last 10 seconds**. A dict `message → last printed time`.

In [3]:
class Logger:
    def __init__(self):
        self.last = {}

    def should_print(self, timestamp, message):
        if message in self.last and timestamp - self.last[message] < 10:
            return False
        self.last[message] = timestamp
        return True

lg = Logger()
print([lg.should_print(t, m) for t, m in [(1, "foo"), (2, "bar"), (3, "foo"), (8, "bar"), (10, "foo"), (11, "foo")]])
# [True, True, False, False, False, True]

[True, True, False, False, False, True]


## 6. Design HashMap from scratch

**Separate chaining:** an array of buckets; each bucket is a list of `(key, value)`. `hash(key) % buckets` picks the bucket. A real implementation **resizes** when the load factor (items / buckets) grows, keeping operations O(1) on average.

In [4]:
class MyHashMap:
    def __init__(self, buckets=1009):                # a prime number spreads keys better
        self.buckets = [[] for _ in range(buckets)]

    def _bucket(self, key):
        return self.buckets[hash(key) % len(self.buckets)]

    def put(self, key, value):
        bucket = self._bucket(key)
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))

    def get(self, key):
        for k, v in self._bucket(key):
            if k == key:
                return v
        return -1

    def remove(self, key):
        bucket = self._bucket(key)
        for i, (k, _) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                return

hm = MyHashMap()
hm.put(1, 1); hm.put(2, 2)
print(hm.get(1), hm.get(3))   # 1 -1
hm.put(2, 1)
print(hm.get(2))   # 1
hm.remove(2)
print(hm.get(2))   # -1

1 -1
1
-1


## 7. Interview thinking

**Say:** "I need O(1) `get`, so a hash map. I also need O(1) random access, so a list as well. The invariant is that the dict maps each value to its index in the list."

**Common mistakes:** deleting from the middle of a list (O(n)), forgetting to update the moved element's index, deleting the dict entry before updating when `val == last`, using binary search on unsorted timestamps.

## Quick Revision

- Pick structures per operation. State the invariant.
- Random + O(1) delete: list + dict, swap with the last.
- Versioned values: sorted timestamps + `bisect_right`.
- Rate limit per message: dict of last time.
- Hash map: buckets + chaining + resizing.